In [1]:
%load_ext autoreload
%autoreload 2

# Co2L: Contrastive Continual Learning

### Explicación de las Fases:
1. **Fase 1 (Representación):** El modelo entrena su "backbone" para aprender a extraer características robustas. Usa aprendizaje contrastivo para agrupar clases similares y la pérdida IRD para mantener las relaciones que aprendió en tareas pasadas (evitando el olvido).
2. **Fase 2 (Clasificación):** El backbone se congela (no cambia). Se entrena una cabeza lineal simple para mapear esas características a las etiquetas de la tarea actual. Al estar congelado el backbone, se protege el conocimiento previo.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from copy import deepcopy
import os
import matplotlib.pyplot as plt

from models import CNN, Co2LModel
from losses import AsymmetricSupConLoss, IRDLoss
from dataloaders import SequentialCIFAR10
from utils import save_co2l_model

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
BATCH_SIZE = 128
NUM_TASKS = 5
print(f"Dispositivo de entrenamiento: {device}")

Dispositivo de entrenamiento: mps


In [3]:
def train_co2l_phase1(model, teacher, train_loader, device, task_id, epochs=10, lr=1e-3, lambda_ird=1.0):
    model.train()
    model.unfreeze_representation()
    
    optimizer = torch.optim.AdamW(
        list(model.backbone.parameters()) + list(model.projection_head.parameters()), 
        lr=lr
    )
    
    criterion_con = AsymmetricSupConLoss(tau=0.07)
    criterion_ird = IRDLoss(kappa=0.07, kappa_star=0.07)
    
    print(f"\n>>> Iniciando Fase 1 (Representación) - Tarea {task_id}")
    
    for epoch in range(epochs):
        running_loss = 0.0
        for (x1, x2), y in train_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            z1, z2 = model.forward_projection(x1), model.forward_projection(x2)
            
            loss_con = criterion_con((z1, z2), y, num_current=x1.shape[0])
            loss_ird = torch.tensor(0.0).to(device)
            if teacher is not None:
                with torch.no_grad():
                    t1, t2 = teacher.forward_projection(x1), teacher.forward_projection(x2)
                loss_ird = criterion_ird(torch.cat([z1, z2], dim=0), torch.cat([t1, t2], dim=0))
            
            loss = loss_con + lambda_ird * loss_ird
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item()
            
        print(f"  Epoch {epoch+1:02d}/{epochs} | Loss Total: {running_loss/len(train_loader):.4f}")
            
    return model

In [4]:
def train_co2l_phase2(model, task_id, train_loader, val_loader, device, epochs=20, lr=0.1):
    model.eval()
    model.freeze_representation()
    
    if not model.classifier.has_task(task_id):
        model.classifier.add_task(task_id, num_classes=2)
    
    optimizer = torch.optim.SGD(model.classifier.heads[str(task_id)].parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    print(f"\n>>> Iniciando Fase 2 (Clasificación) - Tarea {task_id}")
    
    for epoch in range(epochs):
        # Entrenamiento
        model.classifier.heads[str(task_id)].train()
        train_loss, correct_t, total_t = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model.forward_classifier(x, task_id)
            loss = criterion(logits, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss += loss.item()
            _, pred = logits.max(1); total_t += y.size(0); correct_t += pred.eq(y).sum().item()
        
        # Validación
        model.classifier.heads[str(task_id)].eval()
        val_loss, correct_v, total_v = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model.forward_classifier(x, task_id)
                loss = criterion(logits, y)
                val_loss += loss.item()
                _, pred = logits.max(1); total_v += y.size(0); correct_v += pred.eq(y).sum().item()
        
        print(f"  Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss/len(train_loader):.3f} Acc: {100.*correct_t/total_t:5.2f}% | Val Loss: {val_loss/len(val_loader):.3f} Acc: {100.*correct_v/total_v:5.2f}%")
            
    return model

In [5]:
seq_cifar = SequentialCIFAR10(batch_size=BATCH_SIZE, buffer_size=200)
backbone = CNN(in_channels=3, embedding_dim=32)
model = Co2LModel(backbone, embedding_dim=32, proj_dim=128).to(device)

teacher = None

for task_id in range(NUM_TASKS):
    print(f"\n" + "#"*40)
    print(f"# ENTRENANDO TAREA {task_id} #")
    print("#"*40)
    
    # 1. FASE 1
    train_loader_p1 = seq_cifar.get_task_il_two_view_train_loader(task_id, use_buffer=(task_id > 0))
    model = train_co2l_phase1(model, teacher, train_loader_p1, device, task_id, epochs=10)
    
    # 2. FASE 2
    train_loader_p2, val_loader_p2 = seq_cifar.get_task_il_train_val_loaders(task_id, use_buffer=False)
    model = train_co2l_phase2(model, task_id, train_loader_p2, val_loader_p2, device, epochs=20)
    
    # 3. EVALUACIÓN Y BUFFER
    teacher = deepcopy(model).eval()
    print(f"\n>>> Evaluación Final Task-IL (Tarea {task_id})")
    test_loaders = seq_cifar.get_task_il_test_loaders(task_id)
    for tid, loader in test_loaders.items():
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model.forward_classifier(x, tid)
                _, pred = logits.max(1); total += y.size(0); correct += pred.eq(y).sum().item()
        print(f"  Accuracy Tarea {tid}: {100.*correct/total:.2f}%")
    
    seq_cifar.update_buffer(task_id)
    os.makedirs("checkpoints/co2l", exist_ok=True)
    save_co2l_model(model, f"checkpoints/co2l/task_{task_id}.pth")


########################################
# ENTRENANDO TAREA 0 #
########################################

>>> Iniciando Fase 1 (Representación) - Tarea 0


/Users/mateocostantini/Documents/AA_Udesa/5año/1erCuatri/Vision Artificial Avanzada/vision-avanzada-continual-learning/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  Epoch 01/10 | Loss Total: 5.4189
  Epoch 02/10 | Loss Total: 5.3379
  Epoch 03/10 | Loss Total: 5.2856
  Epoch 04/10 | Loss Total: 5.2479
  Epoch 05/10 | Loss Total: 5.2394
  Epoch 06/10 | Loss Total: 5.2070
  Epoch 07/10 | Loss Total: 5.2030
  Epoch 08/10 | Loss Total: 5.1840
  Epoch 09/10 | Loss Total: 5.1715
  Epoch 10/10 | Loss Total: 5.1673

>>> Iniciando Fase 2 (Clasificación) - Tarea 0
  Epoch 01/20 | Train Loss: 0.256 Acc: 90.04% | Val Loss: 0.219 Acc: 91.00%
  Epoch 02/20 | Train Loss: 0.207 Acc: 91.77% | Val Loss: 0.214 Acc: 91.50%
  Epoch 03/20 | Train Loss: 0.210 Acc: 91.77% | Val Loss: 0.231 Acc: 90.80%
  Epoch 04/20 | Train Loss: 0.203 Acc: 91.86% | Val Loss: 0.210 Acc: 91.40%
  Epoch 05/20 | Train Loss: 0.201 Acc: 91.86% | Val Loss: 0.199 Acc: 91.70%
  Epoch 06/20 | Train Loss: 0.203 Acc: 91.86% | Val Loss: 0.198 Acc: 92.70%
  Epoch 07/20 | Train Loss: 0.204 Acc: 91.82% | Val Loss: 0.205 Acc: 92.00%
  Epoch 08/20 | Train Loss: 0.200 Acc: 92.08% | Val Loss: 0.220 Acc: 9